# Associo: Quick Start

This notebook demonstrates the basic usage of **associo** — a high-performance association analysis library built on Polars.

We'll cover:
1. Direct associations (pre-paired lhs/rhs data)
2. Combinatorial associations (market basket style)
3. Selecting specific measures
4. Working with the `all_pairs` option

In [ ]:
import polars as pl
from associo import direct_associations, combinatorial_associations, ALL_MEASURES

## 1. Direct Associations

Use `direct_associations` when your data already has paired columns (e.g., `category → product`, `search_query → clicked_item`).

In [ ]:
# Example: which product categories lead to which product purchases?
orders = pl.DataFrame({
    "category": ["dairy", "dairy", "bakery", "bakery", "dairy", "meat", "meat", "bakery"],
    "product":  ["milk",  "butter", "bread", "rolls", "cheese", "chicken", "beef", "bread"],
    "order_id": [1,       1,        2,       2,       3,        3,         4,      4],
})

orders

In [ ]:
result = direct_associations(
    orders,
    column_lhs="category",
    column_rhs="product",
    column_tid="order_id",
)

# Show key metrics
result.select("lhs", "rhs", "lhs_rhs_count", "support", "confidence", "lift", "conviction", "jaccard")

## 2. Combinatorial Associations

Use `combinatorial_associations` when you have items in transactions and want to find all pairwise associations (classic market basket analysis).

In [ ]:
# Each row = one item in a transaction
baskets = pl.DataFrame({
    "product": [
        "bread", "butter", "milk",       # order 1
        "bread", "milk", "eggs",          # order 2
        "butter", "eggs", "cheese",       # order 3
        "bread", "butter", "milk", "eggs", # order 4
    ],
    "order_id": [
        1, 1, 1,
        2, 2, 2,
        3, 3, 3,
        4, 4, 4, 4,
    ],
})

baskets

In [ ]:
result = combinatorial_associations(
    baskets,
    column_items="product",
    column_tid="order_id",
)

# Top associations by lift
(
    result
    .select("lhs", "rhs", "lhs_rhs_count", "support", "confidence", "lift", "zhangs_metric")
    .sort("lift", descending=True)
    .head(10)
)

## 3. Available Measures

Associo computes 60+ association measures. You can see all of them or select a subset.

In [ ]:
print(f"Total measures available: {len(ALL_MEASURES)}")
print()
for m in sorted(ALL_MEASURES):
    print(f"  {m}")

In [ ]:
# Compute only specific measures using association_measures directly
from associo import association_measures

# First, prepare count data
counts = pl.DataFrame({
    "lhs": ["bread", "milk"],
    "rhs": ["butter", "eggs"],
    "lhs_rhs_count": [50, 30],
    "lhs_total_count": [100, 80],
    "rhs_total_count": [60, 40],
    "total_count": [200, 200],
})

# Compute only selected measures
result = association_measures(
    counts,
    measures=["support", "confidence", "lift", "jaccard", "odds_ratio"],
)

result

## 4. The `all_pairs` Option

By default, only co-occurring pairs are returned. Set `all_pairs=True` to include pairs that never co-occur (with `lhs_rhs_count=0`).

In [ ]:
# Without all_pairs: only pairs that actually appear together
result_default = combinatorial_associations(
    baskets,
    column_items="product",
    column_tid="order_id",
)
print(f"Co-occurring pairs only: {len(result_default)} rows")

# With all_pairs: includes pairs with zero co-occurrences
result_all = combinatorial_associations(
    baskets,
    column_items="product",
    column_tid="order_id",
    all_pairs=True,
)
print(f"All pairs (including zero co-occurrence): {len(result_all)} rows")

# Show pairs with zero co-occurrence
(
    result_all
    .filter(pl.col("lhs_rhs_count") == 0)
    .select("lhs", "rhs", "lhs_rhs_count", "support", "confidence", "lift")
)